# Step 2 — Exploratory Data Analysis (EDA)
### Credit Card Underwriting Pipeline

---

## What is EDA and Why Do We Do It?

> **EDA is like a doctor's first examination before running any tests.**
> Before prescribing treatment (building a model), you must understand the patient (the data).

EDA answers these essential questions:
1. **Shape** — How many rows and columns? Is this enough data?
2. **Types** — Which columns are numeric, which are categorical?
3. **Target** — Is the dataset balanced? (Equal approvals and declines?)
4. **Distributions** — Are features normally distributed or heavily skewed?
5. **Relationships** — Do any features obviously correlate with approval?
6. **Cardinality** — How many unique values does each categorical column have?

---

## Why EDA Before Modelling?

**Common mistakes when skipping EDA:**
- Training on a column that is actually a target leak (contains future information)
- Not noticing that 90% of a column is missing (useless feature)
- Missing that the target is heavily imbalanced (model learns to just predict majority class)
- Feeding a string column to a model that only accepts numbers (crashes)

> **Golden Rule:** Never build a model before you understand your data.


## 2.0 — Re-run Setup (Required in Every Notebook)

In [ ]:
# ── Imports (repeat from Notebook 1) ─────────────────────────────────────────
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
FIGSIZE = (14, 5)
SEED = 42

DATA_PATH   = 'cc_underwriting_5k_stratified11.csv'
IGNORE_COLS = ['applicant_id', 'target_approved', 'target_credit_limit_assigned']
KEY_NUM     = ['age','annual_income','fico_score','debt_to_income_ratio',
               'credit_utilization_ratio','net_worth','requested_credit_limit',
               'fraud_risk_score','avg_bureau_score','monthly_disposable_income']

print('Setup complete')

## 2.1 — Load the Dataset

> **Why `pd.read_csv`?**
> Our data lives in a CSV (Comma-Separated Values) file — a plain-text format where each
> row is one applicant and each column is one attribute.
> `read_csv` loads it into a **DataFrame**: a 2D table structure with labelled rows and columns,
> the central data structure in all Python data science work.


In [ ]:
# Load the raw CSV into a DataFrame
# df_raw = "raw DataFrame" — we never modify this directly
# We always work on copies so we can return to the original if needed
df_raw = pd.read_csv(DATA_PATH)

# Basic shape check: (rows, columns)
# rows    = number of applicants
# columns = number of features + target columns
print(f'Dataset shape: {df_raw.shape}')
print(f'  Rows    (applicants): {df_raw.shape[0]:,}')
print(f'  Columns (features)  : {df_raw.shape[1]:,}')
print()

# Preview the first 3 rows
# This gives us a quick feel for what the data looks like
print('First 3 rows:')
df_raw.head(3)

## 2.2 — Target Variable Analysis

> **The target column is the most important column in the dataset.**
> Everything else is an input; this is what we are trying to predict.

**What we need to check:**
- What are the possible values? (Yes/No, 0/1, categories?)
- Is it balanced? (Equal numbers of each class?)
- Why imbalance matters: if 95% of applicants are approved, a model that always predicts
  "Approved" gets 95% accuracy while being completely useless for finding bad applicants.


In [ ]:
# Count how many applicants fall into each target class
target_counts = df_raw['target_approved'].value_counts()

print('Target distribution:')
print(target_counts)
print()
print(f'Approval rate  : {target_counts["Yes"] / len(df_raw):.1%}')
print(f'Decline rate   : {target_counts["No"]  / len(df_raw):.1%}')
print(f'Imbalance ratio: {target_counts["Yes"] / target_counts["No"]:.2f}:1')

# Visualise: bar chart + pie chart side by side
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# Bar chart — absolute counts
target_counts.plot(kind='bar', ax=axes[0],
                   color=['#e74c3c', '#2ecc71'], edgecolor='white')
axes[0].set_title('Target: Approved?', fontsize=13, fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

# Annotation: add count labels on each bar
for i, v in enumerate(target_counts):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Pie chart — proportional split
axes[1].pie(target_counts, labels=target_counts.index,
            autopct='%1.1f%%', colors=['#e74c3c', '#2ecc71'],
            startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Class Split (%)', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: Credit Card Approval', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Insight: The dataset is moderately imbalanced (~65% Approved).')
print('We will use SMOTE in Step 6 to address this during training.')

## 2.3 — Column Type Inventory

> **Why separate numeric from categorical?**
> - Numeric columns → can compute mean, std, correlations, use directly in most models
> - Categorical columns → must be encoded (converted to numbers) before modelling
> - Each type requires different EDA, different imputation, different encoding strategies

This is the first step of understanding what preprocessing each column will need.


In [ ]:
# Separate columns by data type
# select_dtypes returns a sub-DataFrame of columns matching the given dtype
numeric_cols = df_raw.select_dtypes(include='number').columns.tolist()
cat_cols     = df_raw.select_dtypes(include=['object', 'string']).columns.tolist()

# Remove ID and target columns — they are not features
numeric_cols = [c for c in numeric_cols if c not in IGNORE_COLS]
cat_cols     = [c for c in cat_cols     if c not in IGNORE_COLS]

print(f'Numeric  features: {len(numeric_cols)}')
print(f'Categorical features: {len(cat_cols)}')
print(f'Total candidate features: {len(numeric_cols) + len(cat_cols)}')
print()

# Show all column names grouped by type
print('NUMERIC columns (first 20):')
for c in numeric_cols[:20]:
    print(f'  {c}')

print()
print('CATEGORICAL columns:')
for c in cat_cols:
    print(f'  {c}  (unique values: {df_raw[c].nunique()})')

## 2.4 — Descriptive Statistics for Key Numeric Features

> **Descriptive statistics give you a statistical fingerprint of each column.**
>
> What to look for:
> - **mean vs median far apart** → skewed distribution (outliers present)
> - **min = 0, max = very large** → likely right-skewed (needs log transform later)
> - **std very large relative to mean** → high variability (might need scaling)
> - **min < 0 where impossible** → data quality issue (e.g., negative income)


In [ ]:
# describe() computes count, mean, std, min, 25%, 50%, 75%, max for each column
# .T transposes it (features as rows) for easier reading when there are many columns
stats = df_raw[KEY_NUM].describe().T
print('Descriptive Statistics for Key Numeric Features:')
print()
# Round for cleaner display
stats_rounded = stats.round(2)
print(stats_rounded.to_string())

# Highlight observations
print()
print('Key observations:')
print(f'  age range         : {df_raw["age"].min():.0f} to {df_raw["age"].max():.0f}')
print(f'  annual_income max : ${df_raw["annual_income"].max():,.0f}')
print(f'  fico_score range  : {df_raw["fico_score"].min():.0f} to {df_raw["fico_score"].max():.0f}')
print(f'  net_worth min     : ${df_raw["net_worth"].min():,.0f} (negative = liabilities > assets)')

## 2.5 — Distribution Plots for Key Numeric Features

> **Histograms reveal the shape of each feature's distribution.**
>
> Shapes and what they mean:
> - **Bell curve (normal)** → well-behaved, no transformation needed
> - **Right skew (long tail right)** → log transform recommended (e.g., income)
> - **Left skew** → rare in financial data, may indicate capping
> - **Bimodal (two peaks)** → possibly two distinct customer segments
> - **Spike at zero** → many zero-value records (e.g., people with no savings)


In [ ]:
# Create a 2x5 grid of histograms — one per key numeric feature
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()   # flatten from 2D to 1D array for easy iteration

for i, col in enumerate(KEY_NUM):
    # dropna(): exclude missing values from histogram (don't count them as zeros)
    df_raw[col].dropna().plot(
        kind='hist', bins=40, ax=axes[i],
        color='steelblue', edgecolor='white', alpha=0.8
    )
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    # Add mean line for reference
    mean_val = df_raw[col].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=1.2, alpha=0.7)

plt.suptitle('Distribution of Key Numeric Features  (red dashed = mean)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Observation: annual_income and net_worth are strongly right-skewed.')
print('We will apply log1p() transform in Step 5 to reduce skew.')

## 2.6 — Feature Distribution vs Target

> **The most powerful EDA question: does this feature differ between Approved and Declined?**
>
> If a feature has very similar distributions for both classes → it is likely weak
> If distributions are well separated → it is a strong predictor
>
> This is a visual preview of what the IV analysis will quantify in Step 5.


In [ ]:
# Encode target temporarily for plotting
df_plot = df_raw.copy()
df_plot['target_num'] = (df_plot['target_approved'] == 'Yes').astype(int)

# Box plots: show median, IQR, and outliers for each class side by side
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

plot_features = ['fico_score', 'annual_income', 'debt_to_income_ratio',
                 'credit_utilization_ratio', 'net_worth', 'fraud_risk_score']

colors = {'No': '#e74c3c', 'Yes': '#2ecc71'}

for i, col in enumerate(plot_features):
    # Group by target_approved, plot the feature
    groups = [
        df_plot[df_plot['target_approved'] == 'No'][col].dropna(),
        df_plot[df_plot['target_approved'] == 'Yes'][col].dropna()
    ]
    bp = axes[i].boxplot(groups, patch_artist=True,
                          labels=['Declined', 'Approved'])
    # Colour the boxes
    bp['boxes'][0].set_facecolor('#e74c3c')
    bp['boxes'][1].set_facecolor('#2ecc71')
    for patch in bp['boxes']:
        patch.set_alpha(0.7)

    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Value')

plt.suptitle('Feature Distributions by Approval Status',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Observations:')
print('  fico_score: clearly higher for Approved — strong predictor')
print('  annual_income: slightly higher for Approved')
print('  debt_to_income_ratio: higher for Declined — negative predictor')
print('  fraud_risk_score: higher for Declined')

## 2.7 — Categorical Feature Cardinality

> **Cardinality = number of unique values in a categorical column**
>
> Why it matters:
> - **Low cardinality (2-10)** → great for one-hot encoding
> - **Medium cardinality (10-30)** → use label encoding or target encoding
> - **High cardinality (50+)** → risky for one-hot (too many columns); consider grouping
> - **Very high cardinality (like applicant_id)** → this is an identifier, never a feature


In [ ]:
# Summarise cardinality and top values for all categorical features
print(f'{"Column":<40} {"Unique":>6}  Top Values')
print('-' * 80)

for col in cat_cols:
    n_unique = df_raw[col].nunique()
    # Get the top 3 most frequent values
    top_vals = df_raw[col].value_counts().head(3).index.tolist()
    top_str  = ', '.join(str(v) for v in top_vals)
    print(f'{col:<40} {n_unique:>6}  {top_str}')

## Step 2 Complete — EDA Summary

**What we discovered:**

| Finding | Impact on Pipeline |
|---------|-------------------|
| 4,480 rows × 200 columns | Enough data; need feature selection to manage 200 features |
| ~65% Approved, ~35% Declined | Moderate imbalance → apply SMOTE before training |
| `annual_income`, `net_worth` strongly right-skewed | Log-transform these in Feature Engineering |
| `fico_score` clearly separates classes | Will likely be a top feature by IV |
| `debt_to_income_ratio` higher for Declined | Negative relationship with approval |
| Categorical columns have cardinality 4–50 | Suitable for label encoding |

**Next:** `03_Missing_Values.ipynb` — Detect and treat missing data
